# FROG — etapa 5: planificador, agentes y evaluación

Orden: (0) preparación · (1) pruebas · (2) E2 (CPU) · (3) E3 con Qwen y Phi (GPU) · (4) agente de historia (GPU, requiere las anotaciones de Bihonda).

Entorno: *Entorno de ejecución → Cambiar tipo → T4 GPU* (L4/A100 si hay Colab Pro). Los LLM se cargan de a uno y se liberan al terminar.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 0. Preparación

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Busca la carpeta FROG/codigo en tu Drive (directa o como acceso directo en "Mi unidad").
import glob
_c = (glob.glob('/content/drive/MyDrive/FROG/codigo') +
      glob.glob('/content/drive/.shortcut-targets-by-id/*/FROG/codigo') +
      glob.glob('/content/drive/MyDrive/planif_proyecto/codigo'))     # carpeta antigua
assert _c, 'No encuentro FROG/codigo: agrega la carpeta FROG a "Mi unidad" (clic derecho > Organizar > Agregar acceso directo)'
RAIZ = _c[0]
%cd {RAIZ}


In [ ]:
!pip install -q -r requirements.txt

## 1. Pruebas (CPU, sin modelos)

In [ ]:
!python -m pytest -q tests

## 2. E2: simulación del planificador (CPU, segundos)
500 usuarios simulados, Thompson frente a aleatorio. Deja `salidas/e2/` y las sesiones simuladas que usa E3.

In [ ]:
print(open('salidas/e2/resumen.json').read())

## 3. E3: resúmenes del agente explicador (GPU)
Primero una prueba corta con 3 sesiones; si termina bien, la completa (30 sesiones por LLM).

In [ ]:
!python -m experimentos.e3_resumenes --llm qwen2.5-7b phi-3.5-mini --n 3 --salida salidas/e3_prueba

In [ ]:
!python -m experimentos.e3_resumenes --llm qwen2.5-7b phi-3.5-mini --n 30
print(open('salidas/e3/e3_resumen.json').read())

**Pendiente manual (Nayeli, sin IA):** en `salidas/e3/e3_resumenes.csv`, columna `fiel_segun_revision_manual` (`si`/`no`): ¿el resumen describe fielmente las decisiones, incluida la parte interpretativa ("prefieres colores cálidos")?

## 4. Agente de historia (GPU)
Requiere `datos/d4_historias/anotacion_30.csv` y `datos/d4es_historias/anotacion_30_es.csv` con las columnas `personajes` y `objetos` llenas.

In [ ]:
!python -m experimentos.agente_historia --llm qwen2.5-7b phi-3.5-mini
print(open('salidas/agente_historia/resumen.json').read())

## 5. Prueba rápida de una historia propia

In [ ]:
from portadas.agentes.llm import LLMLocal
from portadas.agentes.historia import extraer_elementos
import json
llm = LLMLocal('qwen2.5-7b')
historia = 'Un conejo que ama las flores las cuida en su jardín durante el invierno.'
print(json.dumps(extraer_elementos(historia, llm), indent=1, ensure_ascii=False))
llm.liberar()